In [7]:
!pip install scikit-learn
!pip install xgboost


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 4.3 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

print('Loading the final dataset..')
file_path = '../data/processed/model_training_data.csv'
df=pd.read_csv(file_path)

df['date']=pd.to_datetime(df['date'])

print(f"Data loaded successfully! Total rows available for training: {len(df)}")
display(df.head())


Loading the final dataset..
Data loaded successfully! Total rows available for training: 57104


,date,scheme_name,direct_nav,regular_nav,nav_difference,pct_drag,return_7d,return_30d,return_90d
0,2019-04-01,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.97,13.31,0.66,4.958678,1.526163,3.023599,2.344322
1,2019-04-02,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,14.02,13.36,0.66,4.940120,1.373825,3.392330,3.012491
2,2019-04-03,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.99,13.33,0.66,4.951238,0.937951,3.171091,3.247232
3,2019-04-04,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.98,13.32,0.66,4.954955,0.431034,2.342606,3.173432
4,2019-04-05,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,13.99,13.32,0.67,5.030030,0.143164,2.116788,3.247232


In [9]:
import pandas as pd

print("Step 1: Extracting the Fund House identity...")

# 1. Create an empty list to hold the extracted corporate names
extracted_fund_houses = []

# 2. Loop through every single scheme name in our dataset explicitly
for name in df['scheme_name']:
    
    # Split the name into a list of individual words 
    # Example: "AXIS SMALL CAP FUND" becomes ['AXIS', 'SMALL', 'CAP', 'FUND']
    words = name.split()
    
    # Grab the very first word (Index 0)
    first_word = words[0]
    
    # Save it to our list
    extracted_fund_houses.append(first_word)

# 3. Add this list as a brand new column to our master table
df['fund_house'] = extracted_fund_houses

print(f"✅ Extracted {len(df['fund_house'].unique())} unique Asset Management Companies!")
display(df[['scheme_name', 'fund_house']].head())

Step 1: Extracting the Fund House identity...
✅ Extracted 17 unique Asset Management Companies!


,scheme_name,fund_house
0,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,ADITYA
1,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,ADITYA
2,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,ADITYA
3,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,ADITYA
4,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,ADITYA


In [10]:
print("Step 2: Translating Corporate Names into 1s and 0s for the AI...")

# 4. Use get_dummies to turn the text column into individual Yes/No (1 or 0) columns
# We set dtype=int so it outputs standard 1s and 0s instead of True/False
df = pd.get_dummies(df, columns=['fund_house'], dtype=int)

print("✅ Translation Complete! Look at how the AI sees the companies now:")

# 5. Let's look at the new columns we just created to verify
# We filter to show the scheme name and any column that starts with 'fund_house_'
amc_columns = [col for col in df.columns if 'fund_house_' in col]
display(df[['scheme_name'] + amc_columns].head())

Step 2: Translating Corporate Names into 1s and 0s for the AI...
✅ Translation Complete! Look at how the AI sees the companies now:


,scheme_name,fund_house_ADITYA,fund_house_AXIS,fund_house_BARODA,fund_house_CANARA,fund_house_DSP,fund_house_EDELWEISS,fund_house_GROWW,fund_house_IDFC,fund_house_INVESCO,fund_house_KOTAK,fund_house_L&T,fund_house_MIRAE,fund_house_NIPPON,fund_house_SBI,fund_house_SUNDARAM,fund_house_UNION,fund_house_UTI
0,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,ADITYA BIRLA SUN LIFE EQUITY SAVINGS FUND,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [11]:
print("Step 3: Calculating the 'Time Alive' compounding tracker...")

# 1. Sort the data chronologically so time flows correctly
df = df.sort_values(by=['scheme_name', 'date'])

# 2. Find the very first date (the launch date) for each individual fund
# .transform('min') finds the oldest date for 'ADITYA' and pastes it on every 'ADITYA' row
df['start_date'] = df.groupby('scheme_name')['date'].transform('min')

# 3. Calculate the exact number of days between the current row's date and the start date
# .dt.days strips away the word "days" so we are left with a pure integer for the AI
df['days_since_launch'] = (df['date'] - df['start_date']).dt.days

# 4. We don't need the temporary start_date column anymore, so drop it to keep things clean
df = df.drop(columns=['start_date'])

print("✅ Compounding tracker successfully built!")
display(df[['date', 'scheme_name', 'days_since_launch', 'pct_drag']].tail(10))

Step 3: Calculating the 'Time Alive' compounding tracker...
✅ Compounding tracker successfully built!


,date,scheme_name,days_since_launch,pct_drag
57094,2020-08-29,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,516,7.192499
57095,2020-08-30,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,517,7.192499
57096,2020-08-31,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,518,7.192997
57097,2020-09-01,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,519,7.193120
57098,2020-09-02,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,520,7.193651
57099,2020-09-03,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,521,7.241913
57100,2020-09-04,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,522,7.242221
57101,2020-09-05,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,523,7.242221
57102,2020-09-06,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,524,7.242221
57103,2020-09-07,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,525,7.242443


In [12]:
print("Step 4: Calculating 90-Day Risk Scores (Volatility)...")

# 1. Create a container for our risk calculations
risk_calculated_pieces = []

# 2. Process one fund at a time
for scheme_name, group_data in df.groupby('scheme_name'):
    
    # Ensure chronological order
    group_data = group_data.sort_values('date')
    
    # 3. Calculate the Risk Score
    # We look at the 'return_7d' column and calculate its standard deviation over 90 days.
    # High standard deviation = High Risk/Volatility. Low standard deviation = Safe/Stable.
    group_data['risk_90d'] = group_data['return_7d'].rolling(window=90).std()
    
    # Save this completed piece
    risk_calculated_pieces.append(group_data)

# 4. Merge everything back into our master table
df = pd.concat(risk_calculated_pieces, ignore_index=True)

# 5. Drop the blank rows created because the rolling window needs 90 days to warm up
df = df.dropna(subset=['risk_90d'])

print(f"✅ Risk Scores generated! Total clean rows remaining: {len(df)}")
display(df[['date', 'scheme_name', 'return_7d', 'risk_90d', 'pct_drag']].tail(10))

Step 4: Calculating 90-Day Risk Scores (Volatility)...
✅ Risk Scores generated! Total clean rows remaining: 53168


,date,scheme_name,return_7d,risk_90d,pct_drag
57074,2020-08-29,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,1.173445,1.629622,7.192499
57075,2020-08-30,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,1.173445,1.565059,7.192499
57076,2020-08-31,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,-2.160482,1.531393,7.192997
57077,2020-09-01,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,-2.098097,1.536648,7.193120
57078,2020-09-02,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,-1.857398,1.563766,7.193651
57079,2020-09-03,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,-1.980229,1.586290,7.241913
57080,2020-09-04,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,-3.050007,1.634119,7.242221
57081,2020-09-05,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,-3.050007,1.677346,7.242221
57082,2020-09-06,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,-3.050007,1.729323,7.242221
57083,2020-09-07,UTI FOCUSSED EQUITY FUND SERIES - 1 (2195 DAYS...,-0.155918,1.734503,7.242443


In [13]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

print("Step 1: Assembling the new 'Business Logic' Feature Set...")

# Automatically grab all 17 of the one-hot encoded AMC columns
amc_columns = [col for col in df.columns if 'fund_house_' in col]

# Combine our momentum features with our brand new structural features
features_to_use = ['return_7d', 'return_30d', 'return_90d', 'days_since_launch', 'risk_90d'] + amc_columns

X = df[features_to_use]
y = df['pct_drag']

print("Step 2: Re-splitting the data...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Step 3: Training the V2 XGBoost Model...")
xgb_model_v2 = xgb.XGBRegressor(
    n_estimators=100,      
    learning_rate=0.1,     
    random_state=42        
)
xgb_model_v2.fit(X_train, y_train)

print("Step 4: Grading the V2 pop quiz...")
predictions_v2 = xgb_model_v2.predict(X_test)
error_v2 = mean_absolute_error(y_test, predictions_v2)
score_v2 = r2_score(y_test, predictions_v2)

print("--------------------------------------------------")
print(f"✅ V2 XGBoost Training Complete!")
print(f"Old R2 Score: 0.0386")
print(f"New Model Score (R2): {score_v2:.4f}")
print(f"Average Error: {error_v2:.4f}%")

Step 1: Assembling the new 'Business Logic' Feature Set...
Step 2: Re-splitting the data...
Step 3: Training the V2 XGBoost Model...
Step 4: Grading the V2 pop quiz...
--------------------------------------------------
✅ V2 XGBoost Training Complete!
Old R2 Score: 0.0386
New Model Score (R2): 0.8886
Average Error: 0.6941%


In [14]:

print("Ranking the most important features driving Commission Drag...")

# 1. Extract the internal importance scores from the XGBoost model
importances = xgb_model_v2.feature_importances_

# 2. Match the scores to our column names
importance_df = pd.DataFrame({
    'Feature': features_to_use,
    'Importance': importances
})

# 3. Sort them from most important to least important
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# 4. Display the top 10 drivers
display(importance_df.head(10))

Ranking the most important features driving Commission Drag...


,Feature,Importance
13,fund_house_INVESCO,0.128383
20,fund_house_UNION,0.116980
16,fund_house_MIRAE,0.109528
8,fund_house_CANARA,0.090536
10,fund_house_EDELWEISS,0.087796
5,fund_house_ADITYA,0.084092
7,fund_house_BARODA,0.067749
17,fund_house_NIPPON,0.055650
21,fund_house_UTI,0.035646
15,fund_house_L&T,0.034536


In [15]:
import joblib
import os

print("Exporting the trained model and features...")

# 1. Save the model
joblib.dump(xgb_model_v2, '../data/processed/xgboost_commission_model.pkl')

# 2. Save the exact column order the model expects
joblib.dump(features_to_use, '../data/processed/model_features.pkl')

# 3. Save the list of unique fund houses so our dropdown menu has the right options
fund_houses_list = [col.replace('fund_house_', '') for col in amc_columns]
joblib.dump(fund_houses_list, '../data/processed/fund_houses.pkl')

print("✅ Model successfully exported! Ready for Streamlit.")

Exporting the trained model and features...
✅ Model successfully exported! Ready for Streamlit.


--------------------------------------------------
✅ Training Complete!
Average Error: The model's guess is off by 2.5503% on average.
Model Score (R2): 0.0002


Step 4: Grading the quiz...
--------------------------------------------------
✅ XGBoost Training Complete!
Average Error: 2.4917% (Linear Regression was 2.5503%)
Model Score (R2): 0.0386 (Linear Regression was 0.0002)
